# The Slacker Network: How Distributed Computing Works

---

## The Setup: One Computer vs. The Internet

Every second, Google processes **~8.5 billion searches**. Netflix streams to **238 million subscribers** simultaneously. ChatGPT answers millions of requests in parallel.

**Could a single computer handle that?** Not even close.

The secret weapon behind all of it: **Distributed Computing** — splitting one massive job across hundreds or thousands of machines, each doing a small piece, then combining the results.

```
One Computer (Single Node)          Distributed System (Many Nodes)
──────────────────────────          ─────────────────────────────────
                                    ┌──────────┐
  ┌────────────────────┐            │  Node A  │── Part 1 of job
  │  ALL the work here │    vs.     │  Node B  │── Part 2 of job
  │  (bottleneck!)     │            │  Node C  │── Part 3 of job
  └────────────────────┘            │  Node D  │── Part 4 of job
                                    └──────────┘
        4 hours                         1 hour  ← combine results
```

> **Key Idea:** A distributed system is a network of independent machines that *coordinate* to solve a problem that's too big for any one of them.

---

## The 3 Core Ideas

### 1. Partitioning — Splitting the Data

Before anything runs, the data gets divided into chunks and assigned to different nodes.

```
Input: ["apple","banana","apple","cherry","banana","apple"]
                          ▼  Partition
Node 1 gets: ["apple", "banana"]
Node 2 gets: ["apple", "cherry"]
Node 3 gets: ["banana", "apple"]
```

### 2. MapReduce — The Two-Step Formula

Almost all distributed workloads follow this pattern:

| Phase | What it does | Analogy |
|-------|-------------|--------|
| **Map** | Each node processes its chunk independently | Each slacker counts their own pile of words |
| **Reduce** | One node combines all partial results | One slacker totals everyone's counts |

```
MAP phase (nodes work in parallel):
  Node 1: {apple:1, banana:1}
  Node 2: {apple:1, cherry:1}
  Node 3: {banana:1, apple:1}
                  ▼
REDUCE phase (combine):
  Final:  {apple:3, banana:2, cherry:1}
```

### 3. Fault Tolerance — When Nodes Crash

In a cluster of 10,000 machines, **failures are guaranteed**. Distributed systems are designed to survive them:
- Work is **replicated** across multiple nodes
- A **coordinator** (master node) detects failures and reassigns work
- The job completes even when 10–20% of nodes die mid-task

---

## Live Demo: MapReduce Word Count

MapReduce word count is the "Hello World" of distributed computing. Run the cell below to watch it happen across 4 simulated nodes:

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from collections import Counter

# Dataset split across 4 nodes
corpus = [
    "distributed systems are powerful and scalable distributed systems",
    "nodes communicate over a network and share the workload across nodes",
    "fault tolerance means the system survives node failures and keeps running",
    "mapreduce splits data across nodes and reduces results into one answer"
]

def map_node(node_id, text):
    time.sleep(0.3)  # simulate compute delay
    counts = Counter(text.split())
    print(f"  Node {node_id} done — top 3: {counts.most_common(3)}")
    return counts

print("=" * 55)
print("  DISTRIBUTED MAPREDUCE WORD COUNT")
print("=" * 55)

print("\n[ MAP phase — 4 nodes running in parallel ]")
start = time.time()
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(map_node, i+1, chunk) for i, chunk in enumerate(corpus)]
    partials = [f.result() for f in futures]
print(f"  → All nodes finished in {time.time()-start:.2f}s")

print("\n[ REDUCE phase — coordinator merges all results ]")
final = Counter()
for p in partials:
    final += p

print("\n[ Final Word Counts (top 10) ]")
print("-" * 35)
for word, count in final.most_common(10):
    print(f"  {word:<16} {'█' * count} ({count})")
print("=" * 55)

---

## Fault Tolerance in Action

What happens when a node crashes mid-job? The coordinator detects the failure (via heartbeat timeout) and **reassigns that partition** to a healthy node. Run this to see it:

In [ ]:
import random

def process_node(node_id, data, fail_chance=0.5):
    time.sleep(0.2)
    if random.random() < fail_chance:
        raise ConnectionError(f"Node {node_id} CRASHED (heartbeat timeout)")
    return sum(data)

def coordinator(node_id, data, max_retries=3):
    for attempt in range(max_retries):
        try:
            result = process_node(node_id, data)
            label = "OK" if attempt == 0 else f"OK after {attempt} retry"
            print(f"  Node {node_id}: {label}  result={result}")
            return result
        except ConnectionError as e:
            print(f"  Node {node_id}: {e} → retrying...")
    print(f"  Node {node_id}: GAVE UP — reassigning to spare node")
    return None

random.seed(42)  # fixed so you see retry behavior
partitions = [[1,2,3], [4,5,6], [7,8,9], [10,11,12]]

print("=" * 50)
print("  FAULT TOLERANCE DEMO")
print("  (each node has 50% crash chance)")
print("=" * 50)

results = [coordinator(i+1, p) for i, p in enumerate(partitions)]
valid = [r for r in results if r is not None]

print(f"\n  Partial results: {valid}")
print(f"  Final answer:    {sum(valid)}")
print(f"  (Expected if all succeed: {sum(range(1,13))})")
print("=" * 50)

---

## Distributed vs. Parallel — Don't Mix Them Up

```
Parallel Computing              Distributed Computing
────────────────────            ──────────────────────
Multiple cores/threads          Multiple MACHINES
Shared memory                   Separate memory + network
Fast, tight coordination        Slower, must survive failures
Your laptop's 8 cores           Google's 10,000 servers
```

Both split work — but distributed systems must also handle **network delays**, **partial failures**, and **no shared state**. That's what makes them hard to build.

---

## Where You've Already Used This

| System | How it's distributed |
|--------|---------------------|
| **Google Search** | Your query hits ~1000 machines simultaneously, results merged in <100ms |
| **Netflix** | Video chunks served from the nearest of 10,000+ CDN nodes worldwide |
| **Bitcoin** | Every transaction validated by thousands of independent nodes |
| **ChatGPT** | Inference split across hundreds of GPUs per request |
| **GitHub** | Every `git clone` is a full copy — distributed version control |

---

## Knowledge Check

**Q1:** In MapReduce, which phase runs in **parallel across all nodes**?

<details>
<summary>Reveal Answer</summary>

**Map phase.** Each node independently processes its chunk at the same time. The Reduce phase runs on a coordinator (or fewer nodes) to merge partial results.

</details>

---

**Q2:** Node 3 crashes halfway through a 8-node job. What happens?

- A) The entire job fails  
- B) The coordinator reassigns Node 3's partition to a healthy node  
- C) The job finishes but skips Node 3's data  
- D) All 8 nodes restart  

<details>
<summary>Reveal Answer</summary>

**B.** The coordinator detects the failure via heartbeat timeout and reassigns that partition. No global restart needed.

</details>

---

**Q3:** Why can't distributed nodes share memory the way threads can?

<details>
<summary>Reveal Answer</summary>

Each machine has its **own physical RAM**. There's no way for Node A in one data center to read Node B's memory directly — all communication goes over a network, which is **orders of magnitude slower** and can fail. This is the core constraint that makes distributed systems architecturally different from multi-threading.

</details>

---

## Hack Challenge

Extend the MapReduce demo to compute **average word length** across all nodes.

**Requirements:**
1. Each node computes `(total_char_count, word_count)` for its chunk — Map phase
2. The coordinator sums both values, then divides — Reduce phase
3. Print the average word length

**Bonus:** Add a 5th node that always crashes. Show the coordinator detecting it and skipping its partition, then note what data is lost.

In [ ]:
# Your solution here
